# Vishwakarma Institute of Technology, Pune
## Course: CI3201 - Deep Learning
### Lab Practical 2.2: Multilayer Perceptron (MLP) for Classification
---
**Objective:** Design and implement a Multilayer Perceptron (MLP) for classification of the Wine dataset, and evaluate its performance using accuracy and a confusion matrix.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. Load and Preprocess the Dataset
We use the **Wine Dataset** which contains 13 features (alcohol, malic acid, etc.) to classify 3 different cultivars (classes).

In [ ]:
# Load Dataset
wine = load_wine()
X = wine.data
y = wine.target

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features (Crucial for Neural Networks)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

### 2. Design the Multilayer Perceptron (MLP)
Architecture: Input Layer (13) -> Hidden Layer 1 (64) -> ReLU -> Hidden Layer 2 (32) -> ReLU -> Output Layer (3)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, output_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden2, output_dim)
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        return out

model = MLP(input_dim=13, hidden1=64, hidden2=32, output_dim=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
print(model)

In [ ]:
# 3. Training Loop
epochs = 100
train_losses = []
val_accuracies = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    # Backward and optimize
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    # Validation Accuracy
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_t)
        _, preds = torch.max(test_outputs, 1)
        acc = accuracy_score(y_test_t.cpu(), preds.cpu())
        val_accuracies.append(acc)
        
    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Val Accuracy: {acc*100:.2f}%")

In [ ]:
# 4. Plot Training Progress
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss', color='red')
plt.xlabel('Epochs')
plt.ylabel('Cross Entropy Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label='Validation Accuracy', color='blue')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve')
plt.legend()
plt.show()

In [ ]:
# 5. Evaluate Performance (Confusion Matrix)
model.eval()
with torch.no_grad():
    final_outputs = model(X_test_t)
    _, final_preds = torch.max(final_outputs, 1)
    
cm = confusion_matrix(y_test_t.cpu(), final_preds.cpu())
final_acc = accuracy_score(y_test_t.cpu(), final_preds.cpu())

print(f"\nFinal Test Accuracy: {final_acc * 100:.2f}%")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=wine.target_names, yticklabels=wine.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Wine Classification')
plt.show()